### Exporting the recommendations to a real Spotify playlist

Goals : take the songs from notebook 05 and actually push them into a playlist on my Spotify account.

> Notebook 05 saved `data/metamood_real_songs.csv`, and every row already carries an exact Spotify `track_id`, so I don't need any fuzzy name matching, I can send the exact tracks straight to a playlist.

> One thing to sort out first : all the earlier notebooks used `SpotifyClientCredentials`, which is the **app-only** flow. It's great for reading the catalog but it **can't create playlists** on my account. For that I need the **user authorization** flow (`SpotifyOAuth`), so there's a tiny one-time setup to do :
> 1. go to my app in the [Spotify Developer Dashboard](https://developer.spotify.com/dashboard) (the same app whose Client ID / Secret are already in my `.env`)
> 2. open **Settings -> Redirect URIs** and add exactly `http://127.0.0.1:8888/callback`
> 3. save
>
> The very first time I run the auth cell below, a browser tab opens to approve it, and after that the token is cached so I'm not asked again.

In [10]:
import os
import spotipy
import pandas as pd
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv

load_dotenv()

# Same Client ID / Secret we already use in notebook 01, read from .env
CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

# This must match EXACTLY what you registered in the dashboard (step 2 above)
REDIRECT_URI = "http://127.0.0.1:8888/callback"

# playlist-modify-private lets us create and fill a private playlist.
# Use playlist-modify-public instead if you want the playlist to be public.
auth_manager = SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope="playlist-modify-private",
    # Separate cache file so we don't clobber the app-only token from notebook 01
    cache_path=".cache-user",
)
sp = spotipy.Spotify(auth_manager=auth_manager, requests_timeout=20)

# A quick call that requires a user token. If this prints your name, auth worked.
me = sp.me()
print(f"Authenticated as: {me['display_name']} ({me['id']})")

Authenticated as: Green (7plmwgvr8ya0arrwn6vabo38j)


In [11]:
# Load the recommendations from notebook 05 and turn track_ids into Spotify URIs
recs = pd.read_csv("../data/metamood_real_songs.csv")

# Drop any rows missing a track_id, just in case, and de-duplicate while keeping order
track_ids = recs["track_id"].dropna().drop_duplicates().tolist()
track_uris = [f"spotify:track:{tid}" for tid in track_ids]

print(f"{len(track_uris)} tracks ready to add")
recs[["artists", "track_name"]].head(10)

50 tracks ready to add


,artists,track_name
0,Sufjan Stevens,Fourth of July
1,Yot Club,YKWIM?
2,Billie Eilish,everything i wanted
3,JVKE,golden hour
4,Tom Rosenthal,Lights Are On
5,Billie Eilish,ocean eyes
6,Liana Flores,rises the moon
7,Lord Huron,The Night We Met
8,Billie Eilish;Khalid,lovely (with Khalid)
9,Joji,Glimpse of Us


In [12]:
# Create a fresh private playlist and add the tracks.
PLAYLIST_NAME = "MetaTune Recommendations"
PLAYLIST_DESCRIPTION = "Generated by the MetaTune music recommender (notebook 05)."

playlist = sp.user_playlist_create(
    user=me["id"],
    name=PLAYLIST_NAME,
    public=False,
    description=PLAYLIST_DESCRIPTION,
)

# The API accepts at most 100 tracks per call, so we add in batches to be safe.
for start in range(0, len(track_uris), 100):
    batch = track_uris[start:start + 100]
    sp.playlist_add_items(playlist["id"], batch)

print(f"Added {len(track_uris)} tracks to '{PLAYLIST_NAME}'")
print(f"Open it here: {playlist['external_urls']['spotify']}")

HTTP Error for POST to https://api.spotify.com/v1/users/7plmwgvr8ya0arrwn6vabo38j/playlists with Params: {} returned 403 due to Forbidden


SpotifyException: http status: 403, code: -1 - https://api.spotify.com/v1/users/7plmwgvr8ya0arrwn6vabo38j/playlists:
 Forbidden, reason: None

> And that's it, the playlist now lives on my Spotify account (I find it under **Your Library**) !
>
> Small note : re-running the last cell creates a **new** playlist every time. If I'd rather rebuild the same one, I can save its id and use `sp.playlist_replace_items(playlist_id, track_uris[:100])` instead of creating a new playlist.